# 논문 C 「한국 HSK 10단위 연도별 실행세율 자료의 구축」 — 검증(실행용)

논문 C의 본문·부록이 인용한 수치 84건(세율 DB 행수·미확정·겹침·커버리지, 양허 별표, 품목 상세, 미러, 스크립트가 CSV로 남긴 로그 값, 법령 대조 표본, 부록의 파일 행수·판본 수·캐시 수)을 세 논문 공용 셀 창고 `cache/cells.py`의 §1·§1b 셀로 등록하고 `EXPECT_C`로 대조한다. `research/`에서 위에서 아래로 실행한다(약 1분). 하나라도 어긋나면 마지막 셀이 멈추고, 등록값은 `outputs/논문C_수치.json`, 결과는 `outputs/논문C_검증_결과.json`에 남는다. 빌더는 `cache/build_nb_paperC.py`. 논문 C가 쓴 프로그램의 기록은 같은 폴더의 `program.ipynb`에 있다.

In [1]:
import os, re, json, warnings, glob
import numpy as np, pandas as pd
import duckdb
import statsmodels.api as sm
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 120)
# 이 노트북은 KCSTARIFF/research/ 에서 연다. 무역통계(KCSDB2)는 환경변수 KCSDB2_ROOT 또는 기본 경로에서 읽는다.
A = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(A, "outputs", "panel")):
    up = os.path.dirname(A)
    if up == A: raise FileNotFoundError("research/outputs/panel 을 찾지 못했습니다")
    A = up
O = os.path.join(A, "outputs"); P = os.path.join(O, "panel")
KCSDB2 = os.environ.get("KCSDB2_ROOT", r"C:\Work\Projects\KCSDB2")
KCS = os.path.join(KCSDB2, "data", "processed", "kcsdb.duckdb"); TAR = os.path.join(os.path.dirname(A), "data", "processed", "kcstariff.duckdb")
CHK = {}
def chk(label, value, digits=2):
    """본문 인용 수치를 등록한다. §6에서 문서와 대조."""
    CHK[label] = round(float(value), digits) if isinstance(value, (int, float, np.floating, np.integer)) else value
    return CHK[label]

def fe_ols(df, y, xcols, fe, cluster, tol=1e-9, maxit=200):
    """다중 고정효과 OLS: 교대 사영으로 중심화한 뒤 OLS, 군집 표준오차(cluster)."""
    d = df[[y] + xcols + fe + [cluster]].dropna().copy()
    Z = d[[y] + xcols].to_numpy(dtype=float)
    groups = [d[f].astype("category").cat.codes.to_numpy() for f in fe]
    for _ in range(maxit):
        Z0 = Z.copy()
        for g in groups:
            m = np.zeros((g.max() + 1, Z.shape[1])); n = np.bincount(g)
            np.add.at(m, g, Z); Z = Z - m[g] / n[g][:, None]
        if np.abs(Z - Z0).max() < tol: break
    yv, X = Z[:, 0], Z[:, 1:]
    XtX_inv = np.linalg.pinv(X.T @ X); b = XtX_inv @ X.T @ yv; e = yv - X @ b
    cl = d[cluster].astype("category").cat.codes.to_numpy(); G = cl.max() + 1
    S = np.zeros((G, X.shape[1])); np.add.at(S, cl, X * e[:, None]); meat = S.T @ S
    n, k = X.shape; V = XtX_inv @ meat @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    se = np.sqrt(np.diag(V)); t = b / se
    return pd.DataFrame({"coef": b, "se": se, "t": t, "within_sd": X.std(axis=0)}, index=xcols).assign(n=n, clusters=G)
print("research:", A); print("KCSDB2:", KCSDB2)

research: C:\Work\Projects\KCSTARIFF\research
KCSDB2: C:\Work\Projects\KCSDB2


### §1 자료 — 패널·코드 쌍·처리군·집행 지정 목록(논문 C가 인용하는 쌍·코드·구간 수)

In [2]:
pc = pd.read_parquet(os.path.join(P, "panel_codes.parquet"))
pp = pd.read_parquet(os.path.join(P, "panel_pairs.parquet"))
pairs = pd.read_csv(os.path.join(O, "코드쌍_목록.csv"), dtype={"hs10_high": str, "hs10_low": str})
pairs = pairs[(pairs.확인.fillna("") == "Y") | pairs.source.isin(["별표", "별표(선택)"])].copy()
pairs["src"] = np.where(pairs.source.isin(["별표", "별표(선택)"]), "별표", "규칙")
pairs["agri"] = pairs.hs10_high.str[:2].astype(int) <= 24
treat = pd.read_csv(os.path.join(O, "처리군_목록.csv"), dtype={"hs10": str, "mate_hs10": str, "pred_codes": str})
pre = pd.read_csv(os.path.join(O, "사전세액심사_대상_이력_2016_2026.csv"), dtype=str)
dist = pd.read_csv(os.path.join(O, "유통이력_신고물품_통합_구간_2009_2026.csv"), dtype=str)
fx = pd.read_csv(os.path.join(O, "환율_월별_USDKRW.csv"))
fxy = fx.groupby("year").krw_per_usd.mean()
print(f"panel_codes {len(pc):,} / panel_pairs {len(pp):,} / 쌍 {len(pairs)} (별표 {int((pairs.src=='별표').sum())}, 규칙 {int((pairs.src=='규칙').sum())}) / 처리군 {len(treat)}")
chk("쌍 수", len(pairs), 0); chk("쌍 코드 수", len(set(pairs.hs10_high) | set(pairs.hs10_low)), 0)
chk("쌍 별표", int((pairs.src == "별표").sum()), 0); chk("쌍 규칙", int((pairs.src == "규칙").sum()), 0)
chk("패널 코드 수", pc.hs10.nunique(), 0); chk("패널 원산지 수", pc.stat_cd.nunique(), 0)
for (rv, ty), n in treat.groupby(["rev", "type"]).size().items(): chk(f"처리군 {rv} {ty}", n, 0)
for st, n in treat[treat.type.str.startswith("감시형")].groupby("sub_type").size().items(): chk(f"감시형 sub_type {st}", n, 0)
chk("사전세액심사 코드 수", pre.hs10.nunique(), 0); chk("유통이력 코드 수", dist.hs10.nunique(), 0); chk("유통이력 구간 수", len(dist), 0)
cg = pd.read_csv(os.path.join(O, "패널_연속성_그룹.csv"))
cg1 = cg[cg.before_musd >= 1]
chk("연속성 집단 수", len(cg), 0); chk("연속성 집단(1백만 달러 이상)", len(cg1), 0); chk("연속성 비율 중위", cg1.ratio.median(), 2); chk("연속성 비율 10분위", cg1.ratio.quantile(.1), 2); chk("연속성 비율 90분위", cg1.ratio.quantile(.9), 2)
chk("연속성 총액 비율", cg1.after_musd.sum() / cg1.before_musd.sum(), 2); chk("잔여 통관 최대 개월", cg.residual_months.max(), 0)

panel_codes 639,188 / panel_pairs 1,070,404 / 쌍 444 (별표 358, 규칙 86) / 처리군 1462


8.0

### §1b 자료 수치 — 세율 DB·양허 별표·품목 상세·미러·수집 검증 CSV·법령 대조·부록의 파일 수

In [3]:
# 1b 자료 논문(C)의 수치: 세율 DB와 산출 CSV에서 등록한다
con1 = duckdb.connect(); con1.execute(f"ATTACH '{TAR.replace(os.sep, '/')}' AS tr (READ_ONLY)"); con1.execute(f"ATTACH '{KCS.replace(os.sep, '/')}' AS s (READ_ONLY)")
for t in ["tariff_code", "tariff_rate", "dim_rate_cd", "fct_applied_rate", "dim_origin_regime"]:
    chk(f"자료 {t} 행", con1.sql(f"SELECT count(*) FROM tr.{t}").fetchone()[0], 0)
yc = con1.sql("SELECT year, count(*) n FROM tr.tariff_code GROUP BY 1").df(); chk("자료 연도당 코드 최소", yc.n.min(), 0); chk("자료 연도당 코드 최대", yc.n.max(), 0)
yc2 = con1.sql("SELECT year, count(*) n FROM tr.tariff_code WHERE hs10 <> '2424000000' GROUP BY 1").df(); chk("자료 연도당 코드 최소(이사화물 제외)", yc2.n.min(), 0); chk("자료 연도당 코드 최대(이사화물 제외)", yc2.n.max(), 0)
chk("자료 채움 행", con1.sql("SELECT count(*) FROM tr.tariff_rate WHERE source='fill'").fetchone()[0], 0)
chk("자료 별표 채움 행", con1.sql("SELECT count(*) FROM tr.tariff_rate WHERE source='annex'").fetchone()[0], 0)   # 2017~2019 ITA 품목 WTO C, 2026-09-13
chk("자료 별표 채움 코드(연도당)", con1.sql("SELECT count(DISTINCT hs10) FROM tr.tariff_rate WHERE source='annex' AND year=2017").fetchone()[0], 0)
ar = con1.sql("SELECT * FROM tr.fct_applied_rate").df()
chk("자료 하한 있는 행", int(ar.floor_won_kg.notna().sum()), 0)
z = pd.concat([ar[c].fillna("") for c in ["txt_A", "txt_C", "txt_W2"]], axis=1).apply(lambda r: any(re.search(r"또는\s*0\s*원", str(v)) for v in r), axis=1)
chk("자료 종량 0원 행", int(z.sum()), 0)
for k, lab in [("specific_only", "종량세만"), ("two_annexes", "두 별표"), ("rice_pre_tariffication", "관세화 전 쌀")]: chk(f"자료 미확정 {lab}", int((ar.undetermined_reason == k).sum()), 0)
chk("자료 미확정 합", int(ar.rate_undetermined.sum()), 0); chk("자료 할당·양허 겹침 행", int((ar.r_P3.notna() & ar.r_W2.notna()).sum()), 0)
chk("자료 한-EU 2023 다데기 기타", float(ar[(ar.year == 2023) & (ar.hs10 == "2103909090")].applied_eu.iloc[0]), 2)
cov = con1.sql("""WITH m AS (SELECT yyyymm//100 AS yr, hs10, sum(imp_dlr) v FROM s.fact_trade WHERE imp_dlr > 0 GROUP BY 1, 2)
    SELECT yr, sum(v) tot, sum(CASE WHEN c.hs10 IS NOT NULL THEN v END) in_code, sum(CASE WHEN a.mfn IS NOT NULL THEN v END) in_mfn
    FROM m LEFT JOIN tr.tariff_code c ON c.year = m.yr AND c.hs10 = m.hs10 LEFT JOIN tr.fct_applied_rate a ON a.year = m.yr AND a.hs10 = m.hs10
    WHERE yr BETWEEN 2007 AND 2026 GROUP BY 1 ORDER BY 1""").df()
cov["code_pct"] = cov.in_code / cov.tot * 100; cov["mfn_pct"] = cov.in_mfn / cov.tot * 100; display(cov.round(3))
chk("자료 커버리지 코드 최소", cov.code_pct.min(), 2); chk("자료 커버리지 코드 최대", cov.code_pct.max(), 2); chk("자료 커버리지 실행세율 최소", cov.mfn_pct.min(), 2); chk("자료 커버리지 실행세율 최대", cov.mfn_pct.max(), 2)
by = pd.read_csv(os.path.join(O, "양허관세_별표1_2025.csv"), dtype={"hs10": str})
chk("자료 별표 1의 가 행", int((by.byeolpyo == "1의 가").sum()), 0); chk("자료 별표 1의 나 행", int((by.byeolpyo == "1의 나").sum()), 0)
chk("자료 부분 양허", int((by.partial.notna() & (by.partial.astype(str).str.strip() != "")).sum()), 0)
ho = by[by.higher_of.astype(str).str.lower() == "true"]
chk("자료 종량 대안 코드", ho.hs10.nunique(), 0); chk("자료 종량 대안 코드 가", ho[ho.byeolpyo == "1의 가"].hs10.nunique(), 0); chk("자료 종량 대안 코드 나", ho[ho.byeolpyo == "1의 나"].hs10.nunique(), 0)
fl_ = ho.specific_won_kg / (ho.adval / 100); chk("자료 하한 최소 원", fl_.min(), 0); chk("자료 하한 최대 원", fl_.max(), 0)
ex_ = pd.read_csv(os.path.join(O, "실행세율_원산지별_쌍코드_2012_2026.csv"), dtype={"hs10": str})
chk("자료 원산지별 실행세율 행", len(ex_), 0); chk("자료 원산지별 코드", ex_.hs10.nunique(), 0); chk("자료 원산지별 원산지", ex_.stat_cd.nunique(), 0); chk("자료 협정 적용 몫", (ex_.regime != "MFN").mean() * 100, 1)
chk("자료 일곱 상대 어긋남(파일)", len(pd.read_csv(os.path.join(O, "실행세율_원산지별_검증.csv"))), 0)
dd = pd.read_csv(os.path.join(O, "품목상세_세율_2012_2026.csv"), dtype={"hs10": str, "year": int})
seven = ["FCN1", "FEU1", "FUS1", "FAS1", "FIN1", "FVN1", "FCA1"]
mr = con1.sql("SELECT year, hs10, rate_cd, adval FROM tr.tariff_rate WHERE source='main' AND rate_cd IN ('FCN1','FEU1','FUS1','FAS1','FIN1','FVN1','FCA1') AND month(valid_from)=1 AND day(valid_from)=1").df()
cmp_ = dd[dd.rate_cd.isin(seven)].merge(mr, on=["year", "hs10", "rate_cd"])
chk("자료 일곱 상대 대조 행", len(cmp_), 0); chk("자료 일곱 상대 대조 어긋남", int((cmp_.adval_first.astype(float).round(2) != cmp_.adval.round(2)).sum()), 0)
chk("자료 품목 상세 쪽", len(glob.glob(os.path.join(A, "data", "raw", "clip_item_detail", "*", "*.html"))), 0); chk("자료 품목 상세 세율 행", len(dd), 0)
chk("자료 품목 상세 코드", dd.hs10.nunique(), 0); chk("자료 품목 상세 구분기호", dd.rate_cd.nunique(), 0); chk("자료 선택1 협정", dd[dd.rate_cd.str.match(r"^F[A-Z]+1$")].rate_cd.nunique(), 0)
chk("자료 RCEP 일본 코드", dd[dd.rate_cd == "FRCJP1"].hs10.nunique(), 0)
cm_ = pd.read_csv(os.path.join(O, "comtrade_mirror_hs6_2012_2024.csv"), dtype={"cmdCode": str}); chk("자료 미러 행", len(cm_), 0); chk("자료 미러 보고국", cm_.reporter.nunique(), 0); chk("자료 미러 HS6", cm_.cmdCode.nunique(), 0)
chk("자료 코드 목록", sum(1 for l in open(os.path.join(O, "코드쌍_코드목록.txt"), encoding="utf-8") if l.strip()), 0)
# 스크립트가 로그 대신 CSV로 남긴 검증 수치: 01(수집_검증.csv, --verify-only), 02(실행세율_커버리지.csv), 19(원산지별 검증 요약·수입 몫)
REPO = os.path.dirname(A)
vc = pd.read_csv(os.path.join(REPO, "outputs", "수집_검증.csv"), dtype={"key": str})
for cd in ["A", "C"]:
    chk(f"자료 두 화면 일치율 {cd}", float(vc[vc.item == f"두 화면 일치율 {cd}"].value.iloc[0]), 2); chk(f"자료 두 화면 대조 쌍 {cd}", float(vc[vc.item == f"두 화면 대조 쌍 {cd}"].value.iloc[0]), 0)
chk("자료 숫자로 못 읽은 세율 행", float(vc[vc.item == "숫자로 못 읽은 세율 행"].value.iloc[0]), 0)
cs_ = vc[vc.item == "코드 집합 한쪽에만"].astype({"key": int}).set_index("key").value
chk("자료 코드 집합 한쪽에만 최대(2009~, 2011 제외)", float(cs_[(cs_.index >= 2009) & (cs_.index != 2011)].max()), 0)
for y in [2007, 2008, 2011]: chk(f"자료 코드 집합 한쪽에만 {y}", float(cs_[y]), 0)
pf = vc[vc.item == "수입액 커버리지 코드 있음(채움 전)"].astype({"key": int}); chk("자료 채움 전 커버리지 2011", float(pf[pf.key == 2011].value.iloc[0]), 2); chk("자료 채움 전 커버리지 최소", pf.value.min(), 2)
cv2 = pd.read_csv(os.path.join(REPO, "outputs", "실행세율_커버리지.csv")); chk("자료 02 커버리지 실행세율 최소", cv2.pct_rate.min(), 2); chk("자료 02 커버리지 코드 최소", cv2.pct_code.min(), 2)
vs = pd.read_csv(os.path.join(O, "실행세율_원산지별_검증_요약.csv")).set_index("item").value; chk("자료 19 일곱 상대 대조 행", float(vs["일곱 상대 대조 행"]), 0); chk("자료 19 일곱 상대 어긋남", float(vs["일곱 상대 어긋남"]), 0)
sh = pd.read_csv(os.path.join(O, "수입몫_원산지군_2025.csv")).set_index("group").share_pct
for g in ["일곱 상대", "그 밖 협정", "무협정", "일본"]: chk(f"자료 2025 수입 몫 {g}", float(sh[g]), 1)
# 법령 대조 표본 214개(2026-09-13): 관세법 별표 관세율표(2025-01-01 시행 판본 HWP 파싱)·양허관세 규정 별표 1·조정관세 규정 별표와 대조한 결과(outputs/실행세율_법령대조.csv, 요약 CSV)
lc = pd.read_csv(os.path.join(O, "실행세율_법령대조_요약.csv")).set_index("item").value
for k in ["표본", "확인", "양허 W2 일치", "WTO C 일치", "기본세율 A 일치", "조정관세 L 일치", "기본세율 단일 소호 일치", "기본세율 세부 항목 일치", "어긋남", "관세율표 소호 수", "관세율표 세율 항목 수"]: chk(f"자료 법령 대조 {k}", float(lc[k]), 0)
ld = pd.read_csv(os.path.join(O, "실행세율_법령대조.csv"), dtype=str, keep_default_na=False); chk("자료 법령 대조 빈 열", int((ld.법령_확인세율 == "").sum()), 0)
# 부록 B(재현 절차, 2026-09-13 합본)가 인용하는 보조 자료의 행수·판본 수·캐시 수
EXT = os.path.join(A, "data", "external")
for f_, lab in [("유통이력_신고물품_판본별_2009_2026.csv", "유통이력 판본별 행"), ("유통이력_신고물품_이관후_판본별_2022_2026.csv", "유통이력 이관후 판본별 행"), ("유통이력_신고물품_지정구간_2009_2026.csv", "유통이력 지정구간"), ("유통이력_신고물품_이관후_지정구간_2022_2026.csv", "유통이력 이관후 지정구간"), ("유통이력_신고물품_코드요약_2009_2026.csv", "유통이력 코드요약 행"), ("사전세액심사_대상_월별_2016_2026.csv", "사전세액심사 월별 행"), ("환율_월별_USDKRW.csv", "환율 월 수"), ("분류기준_규칙_별표.csv", "분류기준 별표 항목")]:
    chk(f"자료 {lab}", len(pd.read_csv(os.path.join(O, f_), dtype=str)), 0)
chk("자료 유통이력 관세청 판본 수", pd.read_csv(os.path.join(O, "유통이력_신고물품_판본별_2009_2026.csv"), dtype=str).version_eff.nunique(), 0)
chk("자료 유통이력 이관후 판본 수", pd.read_csv(os.path.join(O, "유통이력_신고물품_이관후_판본별_2022_2026.csv"), dtype=str).version_eff.nunique(), 0)
chk("자료 사전세액심사 기준일 수", pd.read_csv(os.path.join(O, "사전세액심사_대상_월별_2016_2026.csv"), dtype=str).ym.nunique(), 0)
chk("자료 미러 캐시 파일", len(glob.glob(os.path.join(A, "data", "raw", "comtrade", "*.json"))), 0)
chk("자료 채움 코드", con1.sql("SELECT count(DISTINCT hs10) FROM tr.tariff_rate WHERE source='fill'").fetchone()[0], 0)
for f_, lab in [("1-1. 세계무역기구협정 등에 의한 양허관세 규정 별표 1의 가.xlsx", "별표1 가 원본 행"), ("1-2. 세계무역기구협정 등에 의한 양허관세 규정 별표 1의 나.xlsx", "별표1 나 원본 행")]:
    chk(f"자료 {lab}", len(pd.read_excel(os.path.join(EXT, "양허관세_규정_별표1", f_), header=None)), 0)
dl = os.path.join(EXT, "유통이력관리_고시_별표1")
chk("자료 유통이력 별표 파일 관세청", len([f_ for f_ in os.listdir(dl) if f_.endswith((".hwp", ".hwpx", ".pdf"))]), 0); chk("자료 유통이력 별표 파일 농식품부", len(os.listdir(os.path.join(dl, "농식품부"))), 0); chk("자료 유통이력 별표 파일 해수부", len(os.listdir(os.path.join(dl, "해수부"))), 0)
chk("자료 관세율표 HWP 바이트", os.path.getsize(os.path.join(EXT, "관세법_별표_관세율표", "관세율표_개정2022-12-31_시행2025-01-01_lsiSeq17031935.hwp")), 0)

,yr,tot,in_code,in_mfn,code_pct,mfn_pct
0,2007,3.567697e+11,3.566846e+11,3.564883e+11,99.976,99.921
1,2008,4.352111e+11,4.352069e+11,4.351244e+11,99.999,99.980
2,2009,3.230328e+11,3.230222e+11,3.228536e+11,99.997,99.945
3,2010,4.251354e+11,4.251351e+11,4.249560e+11,100.000,99.958
4,2011,5.243389e+11,5.243385e+11,5.241744e+11,100.000,99.969
5,2012,5.195128e+11,5.194837e+11,5.193621e+11,99.994,99.971
6,2013,5.154919e+11,5.154916e+11,5.153646e+11,100.000,99.975
7,2014,5.254544e+11,5.254452e+11,5.253432e+11,99.998,99.979
8,2015,4.361586e+11,4.361585e+11,4.361071e+11,100.000,99.988
9,2016,4.058234e+11,4.058233e+11,4.057889e+11,100.000,99.991


1325568.0

### §6 검증 — 논문 C의 인용 수치 대조

In [4]:
EXPECT_C = {
 "자료 tariff_code 행": (238015, 0), "자료 tariff_rate 행": (2024063, 0), "자료 별표 채움 행": (2466, 0), "자료 별표 채움 코드(연도당)": (822, 0), "자료 dim_rate_cd 행": (24, 0), "자료 fct_applied_rate 행": (237990, 0), "자료 dim_origin_regime 행": (49, 0),
 "자료 연도당 코드 최소(이사화물 제외)": (11293, 0), "자료 연도당 코드 최대(이사화물 제외)": (12243, 0), "자료 채움 행": (902, 0),
 "자료 하한 있는 행": (1092, 0),  # "0원" 113행은 재현되지 않아(현재 정의로 238·230) 논문 C에서 수치 삭제 제안 "자료 미확정 합": (530, 0), "자료 미확정 종량세만": (462, 0), "자료 미확정 두 별표": (20, 0), "자료 미확정 관세화 전 쌀": (48, 0), "자료 할당·양허 겹침 행": (12, 0),
 "자료 한-EU 2023 다데기 기타": (9.79, 0.005), "자료 커버리지 코드 최소": (99.98, 0.005), "자료 커버리지 코드 최대": (100, 0.005), "자료 커버리지 실행세율 최소": (99.92, 0.005), "자료 커버리지 실행세율 최대": (100, 0.005),
 "자료 별표 1의 가 행": (9944, 0), "자료 별표 1의 나 행": (278, 0), "자료 부분 양허": (155, 0),  # 논문 C IV.2: 154 → 155 수정 대기 "자료 종량 대안 코드": (92, 0), "자료 종량 대안 코드 가": (46, 0), "자료 종량 대안 코드 나": (46, 0), "자료 하한 최소 원": (88, 0.5), "자료 하한 최대 원": (34000, 500),
 "자료 원산지별 실행세율 행": (187141, 0), "자료 원산지별 코드": (285, 0), "자료 원산지별 원산지": (63, 0), "자료 협정 적용 몫": (87.3, 0.05), "자료 일곱 상대 대조 행": (22164, 0), "자료 일곱 상대 대조 어긋남": (0, 0), "자료 일곱 상대 어긋남(파일)": (0, 0),
 "자료 두 화면 일치율 A": (99.80, 0.005), "자료 두 화면 일치율 C": (99.97, 0.005), "자료 코드 집합 한쪽에만 최대(2009~, 2011 제외)": (1, 0), "자료 코드 집합 한쪽에만 2007": (3, 0), "자료 코드 집합 한쪽에만 2008": (4, 0), "자료 코드 집합 한쪽에만 2011": (447, 0), "자료 숫자로 못 읽은 세율 행": (0, 0), "자료 채움 전 커버리지 2011": (96.75, 0.005),  # 논문 C: 97.38 → 96.75, "해마다 한 개" → 2009년부터(2007년 3·2008년 4·2011년 채움 전 447) 수정 대기
 "자료 02 커버리지 실행세율 최소": (99.92, 0.005), "자료 02 커버리지 코드 최소": (99.98, 0.005), "자료 19 일곱 상대 대조 행": (22164, 0), "자료 19 일곱 상대 어긋남": (0, 0),
 "자료 2025 수입 몫 일곱 상대": (59.7, 0.05), "자료 2025 수입 몫 그 밖 협정": (18.9, 0.05), "자료 2025 수입 몫 일본": (7.7, 0.05), "자료 2025 수입 몫 무협정": (21.4, 0.05),  # 논문 C VII: 16.7·23.6 → 18.9·21.4 수정 대기
 "자료 품목 상세 쪽": (4275, 0), "자료 품목 상세 코드": (285, 0), "자료 품목 상세 구분기호": (110, 0), "자료 선택1 협정": (32, 0),
 "유통이력 구간 수": (237, 0), "유통이력 코드 수": (155, 0), "사전세액심사 코드 수": (48, 0), "자료 미러 행": (91318, 0), "자료 미러 보고국": (28, 0),
 # VI 법령 대조(2026-09-13)
 "자료 법령 대조 표본": (214, 0), "자료 법령 대조 확인": (214, 0), "자료 법령 대조 빈 열": (0, 0), "자료 법령 대조 양허 W2 일치": (34, 0), "자료 법령 대조 WTO C 일치": (25, 0), "자료 법령 대조 기본세율 A 일치": (153, 0), "자료 법령 대조 조정관세 L 일치": (2, 0),
 "자료 법령 대조 기본세율 단일 소호 일치": (176, 0), "자료 법령 대조 기본세율 세부 항목 일치": (38, 0), "자료 법령 대조 어긋남": (0, 0), "자료 법령 대조 관세율표 소호 수": (5612, 0), "자료 법령 대조 관세율표 세율 항목 수": (6969, 0),
 # 부록 B 재현 절차(2026-09-13 합본)
 "자료 유통이력 판본별 행": (1275, 0), "자료 유통이력 이관후 판본별 행": (310, 0), "자료 유통이력 지정구간": (331, 0), "자료 유통이력 이관후 지정구간": (180, 0), "자료 유통이력 코드요약 행": (135, 0), "자료 유통이력 관세청 판본 수": (37, 0), "자료 유통이력 이관후 판본 수": (11, 0),
 "자료 사전세액심사 기준일 수": (129, 0), "사전세액심사 코드 수": (48, 0), "자료 미러 캐시 파일": (390, 0), "자료 미러 HS6": (245, 0), "자료 채움 코드": (446, 0), "자료 채움 행": (902, 0), "자료 별표1 가 원본 행": (15171, 0), "자료 별표1 나 원본 행": (552, 0), "자료 부분 양허": (155, 0), "자료 종량 대안 코드": (92, 0),
 "자료 유통이력 별표 파일 관세청": (37, 0), "자료 유통이력 별표 파일 농식품부": (9, 0), "자료 유통이력 별표 파일 해수부": (9, 0), "자료 관세율표 HWP 바이트": (1325568, 0), "자료 환율 월 수": (237, 0), "자료 분류기준 별표 항목": (43, 0), "자료 품목 상세 쪽": (4275, 0), "자료 품목 상세 세율 행": (78772, 0), "자료 품목 상세 구분기호": (110, 0),
}
def check(name, exp):
    bad, miss, ok = [], [], 0
    for k, (v, tol) in exp.items():
        if k not in CHK: miss.append(k); continue
        got = CHK[k]
        try: g = float(got)
        except (TypeError, ValueError): g = np.nan
        if not np.isfinite(g) or abs(g - v) > tol + 1e-9: bad.append((k, v, got))
        else: ok += 1
    print(f"[{name}] 대조 {len(exp)}건: 통과 {ok}, 불일치 {len(bad)}, 미등록 {len(miss)}")
    for b in bad: print("   불일치", b)
    for m in miss: print("   미등록", m)
    return bad, miss
RES = {"논문 C": check("논문 C", EXPECT_C)}
cited = set(EXPECT_C)
print("등록만 되고 인용되지 않은 라벨:", len([k for k in CHK if k not in cited]))
json.dump(CHK, open(os.path.join(O, "논문C_수치.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
json.dump({n: dict(bad=[list(map(str, b)) for b in r[0]], miss=r[1]) for n, r in RES.items()}, open(os.path.join(O, "논문C_검증_결과.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
n_bad = sum(len(r[0]) + len(r[1]) for r in RES.values())
assert n_bad == 0, f"논문 대조 실패 {n_bad}건 — outputs/논문C_검증_결과.json"
print("논문 C 대조 통과:", len(EXPECT_C), "건")

[논문 C] 대조 84건: 통과 84, 불일치 0, 미등록 0
등록만 되고 인용되지 않은 라벨: 47
논문 C 대조 통과: 84 건
